# nb_06_silver_premiums

**Layer**: Silver | **Source**: `lh_bronze_insurance.premiums_raw` | **Target**: `lh_silver_insurance.premiums`

**Data Quality Rules**:
- Cast `amount_due`, `amount_paid` to DoubleType; `payment_date` to DateType
- Require non-null: `premium_id`, `policy_id`, `amount_due`
- Deduplicate on `premium_id` (keep latest `_ingested_at`)
- Standardize `payment_status` to lowercase
- Rejects → `premiums_quarantine`

**Idempotency**: Uses `overwrite` mode.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, to_date, current_timestamp, lit, row_number, when, coalesce, lower
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

try:
    spark
except NameError:
    spark = SparkSession.builder.appName("nb_06_silver_premiums").getOrCreate()

print("nb_06_silver_premiums - Silver Layer Transformation")

In [ ]:
# ============================================================
# Step 1: Read from Bronze
# ============================================================
df_bronze = spark.table("premiums_raw")
bronze_count = df_bronze.count()
print(f"Bronze rows read: {bronze_count:,}")
df_bronze.printSchema()

In [ ]:
# ============================================================
# Step 2: Cast types & clean
# ============================================================
df_typed = df_bronze \
    .withColumn("premium_id", trim(col("premium_id"))) \
    .withColumn("policy_id", trim(col("policy_id"))) \
    .withColumn("billing_period", trim(col("billing_period"))) \
    .withColumn("amount_due", col("amount_due").cast(DoubleType())) \
    .withColumn("amount_paid", col("amount_paid").cast(DoubleType())) \
    .withColumn("payment_date", to_date(col("payment_date"), "yyyy-MM-dd")) \
    .withColumn("payment_status", lower(trim(col("payment_status"))))

print("Type casting complete.")

In [ ]:
# ============================================================
# Step 3: Validate & route rejects
# ============================================================
# amount_due becomes null after cast if original was empty string
REQUIRED_FIELDS = ["premium_id", "policy_id"]

rejection_conditions = [
    when((col(f).isNull()) | (col(f) == ""), lit(f"null_{f}"))
    for f in REQUIRED_FIELDS
]
# Special handling for numeric field that may have become null from empty string
rejection_conditions.append(
    when(col("amount_due").isNull(), lit("null_amount_due"))
)

df_validated = df_typed.withColumn("_rejection_reason", coalesce(*rejection_conditions))

df_valid = df_validated.filter(col("_rejection_reason").isNull()).drop("_rejection_reason")
df_rejected = df_validated.filter(col("_rejection_reason").isNotNull())

valid_count = df_valid.count()
rejected_count = df_rejected.count()
print(f"Valid: {valid_count:,} | Rejected: {rejected_count:,}")

In [ ]:
# ============================================================
# Step 4: Deduplicate on premium_id
# ============================================================
window_spec = Window.partitionBy("premium_id").orderBy(col("_ingested_at").desc())

df_deduped = df_valid \
    .withColumn("_row_num", row_number().over(window_spec)) \
    .filter(col("_row_num") == 1) \
    .drop("_row_num")

deduped_count = df_deduped.count()
dupes_removed = valid_count - deduped_count
print(f"After dedup: {deduped_count:,} ({dupes_removed} duplicates removed)")

In [ ]:
# ============================================================
# Step 5: Write Silver & Quarantine
# ============================================================
silver_columns = ["premium_id", "policy_id", "billing_period",
                  "amount_due", "amount_paid", "payment_date", "payment_status"]

df_silver = df_deduped.select(silver_columns) \
    .withColumn("_processed_at", current_timestamp())

df_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("premiums")
print(f"✓ {deduped_count:,} rows → premiums")

if rejected_count > 0:
    df_rejected.withColumn("_quarantined_at", current_timestamp()) \
        .write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable("premiums_quarantine")
    print(f"✓ {rejected_count:,} rows → premiums_quarantine")

In [ ]:
# ============================================================
# Validation
# ============================================================
print("\nSILVER PREMIUMS - SUMMARY")
print("=" * 50)
print(f"  Bronze input:       {bronze_count:>8,}")
print(f"  Rejected:           {rejected_count:>8,}")
print(f"  Duplicates removed: {dupes_removed:>8,}")
print(f"  Silver output:      {deduped_count:>8,}")
print(f"  Quality rate:       {(deduped_count/bronze_count*100):.1f}%")

assert spark.table("premiums").count() == deduped_count
print("\n✓ Verification passed")
spark.table("premiums").show(5, truncate=25)